# 🎯 Notebook 2 — Coreset Sampling (`core/coreset.py`)

**What this file does:** After the feature extractor produces hundreds of thousands of patch vectors, this file picks a small, *representative* subset that still covers the full diversity of the data.

**Why it exists:** Storing every patch (~200k+ per category) would make test-time search extremely slow and eat all our GPU memory. Coreset sampling keeps only ~1% of patches while preserving coverage.

**How it fits:**
```
feature_extractor.py → THIS FILE → memory_bank.py
```

**Algorithm:** Greedy farthest-point sampling — like placing cell towers: always put the next one at the spot farthest from any existing tower, guaranteeing maximum coverage with a limited budget.

## 1 — Setup
Mount Google Drive and configure logging.
This notebook loads the raw feature `.pt` files saved by Notebook 01.

In [ ]:
# ---- Mount Google Drive ----
from google.colab import drive
drive.mount('/content/drive')

# ---- Imports ----
import os, logging, torch
from tqdm import tqdm
from typing import Tuple

# ---- Logging ----
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('coreset')

# ---- Paths ----
DRIVE_OUTPUT   = '/content/drive/MyDrive/defects-detection-CV/outputs'
FEATURES_DIR   = os.path.join(DRIVE_OUTPUT, 'raw_features')
MEMORY_DIR     = os.path.join(DRIVE_OUTPUT, 'memory_banks')
os.makedirs(MEMORY_DIR, exist_ok=True)

CATEGORIES     = ['bottle', 'carpet', 'screw']
CORESET_RATIO  = 0.01          # keep 1% of patches
CHECKPOINT_EVERY = 500         # save progress every 500 iterations
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'

logger.info('Device: %s', DEVICE)
logger.info('Features dir: %s', FEATURES_DIR)
logger.info('Memory banks dir: %s', MEMORY_DIR)

---
## 🎯 `core/coreset.py` — Implementation
This file contains just **2 functions**:
1. `greedy_coreset_sampling` — the core algorithm
2. `subsample_memory_bank` — a convenience wrapper

### Function 1 — `greedy_coreset_sampling`

This is the heart of the compression. Given ~200k patch vectors, it selects
the ~2k most *diverse* ones using **farthest-point sampling**:

1. Start from a random point.
2. Find the point farthest from everything we’ve picked so far.
3. Add it to our selection.
4. Repeat until we hit our budget (1% of total).

Includes **checkpointing** every 500 iterations so you don’t lose progress
if Colab disconnects.

In [ ]:
def greedy_coreset_sampling(
    features: torch.Tensor,
    ratio: float,
    device: str,
    checkpoint_path: str = '',
    checkpoint_every: int = 500,
) -> torch.Tensor:
    """Select a diverse subset via greedy farthest-point sampling.

    Args:
        features: (N, D) tensor of patch vectors (on CPU).
        ratio: Fraction to keep (e.g. 0.01 = 1%).
        device: 'cuda' or 'cpu'.
        checkpoint_path: Where to save progress (empty = no checkpointing).
        checkpoint_every: Save every N iterations.

    Returns:
        Tensor of selected indices, shape (n_select,).
    """
    n: int = features.shape[0]
    n_select: int = max(1, int(n * ratio))
    logger.info('Coreset: keeping %d / %d patches (%.1f%%)', n_select, n, ratio * 100)

    features_gpu: torch.Tensor = features.to(device)

    # --- Try to resume ---
    selected_idx: list = []
    min_distances: torch.Tensor = torch.full((n,), float('inf'), device=device)

    if checkpoint_path:
        try:
            ckpt = torch.load(checkpoint_path, map_location=device, weights_only=True)
            selected_idx = ckpt['selected_idx']
            min_distances = ckpt['min_distances'].to(device)
            logger.info('Resumed coreset at iteration %d', len(selected_idx))
        except FileNotFoundError:
            logger.info('No checkpoint — starting fresh.')
        except Exception as exc:
            logger.warning('Checkpoint load failed (%s) — starting fresh.', exc)

    # Pick random seed if starting fresh
    if len(selected_idx) == 0:
        seed: int = torch.randint(0, n, (1,)).item()
        selected_idx.append(seed)
        diff = features_gpu - features_gpu[seed].unsqueeze(0)
        min_distances = torch.sum(diff ** 2, dim=1)

    # --- Main loop ---
    remaining = n_select - len(selected_idx)
    pbar = tqdm(range(remaining), desc='Coreset sampling',
                initial=len(selected_idx), total=n_select)

    for i in pbar:
        # Pick the point farthest from the selected set
        next_idx: int = torch.argmax(min_distances).item()
        selected_idx.append(next_idx)

        # Update distances
        diff = features_gpu - features_gpu[next_idx].unsqueeze(0)
        distances = torch.sum(diff ** 2, dim=1)
        min_distances = torch.minimum(min_distances, distances)

        # Checkpoint
        if checkpoint_path and (i + 1) % checkpoint_every == 0:
            try:
                torch.save(
                    {'selected_idx': selected_idx,
                     'min_distances': min_distances.cpu()},
                    checkpoint_path,
                )
                logger.info('Checkpoint saved (%d / %d)', len(selected_idx), n_select)
            except Exception as exc:
                logger.warning('Checkpoint save failed: %s', exc)

    pbar.close()

    # Clean up checkpoint
    if checkpoint_path:
        try:
            if os.path.exists(checkpoint_path):
                os.remove(checkpoint_path)
        except Exception as exc:
            logger.warning('Could not remove checkpoint: %s', exc)

    result = torch.tensor(selected_idx, dtype=torch.long)
    logger.info('Coreset done: %d indices selected.', len(result))
    return result

### Function 2 — `subsample_memory_bank`

A simple wrapper around the algorithm above. It:
1. Calls `greedy_coreset_sampling` to get the best indices.
2. Uses those indices to slice the original feature tensor.
3. Returns the smaller tensor on CPU, ready to be saved.

This is what `train.py` actually calls — it doesn’t need to know about
indices or the internal algorithm.

In [ ]:
def subsample_memory_bank(
    features: torch.Tensor,
    ratio: float,
    device: str,
    checkpoint_path: str = '',
    checkpoint_every: int = 500,
) -> torch.Tensor:
    """Apply coreset subsampling and return the reduced memory bank.

    Args:
        features: (N, D) tensor of all patch features.
        ratio: Fraction to keep (e.g. 0.01).
        device: 'cuda' or 'cpu'.
        checkpoint_path: Path for checkpoint file.
        checkpoint_every: Checkpoint interval.

    Returns:
        Reduced tensor of shape (n_select, D) on CPU.
    """
    indices = greedy_coreset_sampling(
        features, ratio, device,
        checkpoint_path=checkpoint_path,
        checkpoint_every=checkpoint_every,
    )
    reduced = features[indices].cpu()
    logger.info('Memory bank subsampled: %s → %s', features.shape, reduced.shape)
    return reduced

---
## ▶️ Run Coreset Sampling
Load each category’s raw features from Notebook 01, subsample them,
and save the compressed memory bank to Google Drive.

In [ ]:
for category in CATEGORIES:
    feat_path = os.path.join(FEATURES_DIR, f'{category}_features.pt')
    save_path = os.path.join(MEMORY_DIR, f'{category}_memory_bank.pt')
    ckpt_path = os.path.join(MEMORY_DIR, f'{category}_coreset_ckpt.pt')

    # Skip if already done
    if os.path.exists(save_path):
        logger.info('[%s] Memory bank already exists — skipping.', category)
        continue

    # Load raw features
    logger.info('=== Processing: %s ===', category)
    try:
        features = torch.load(feat_path, map_location='cpu', weights_only=True)
        logger.info('[%s] Loaded features: %s', category, features.shape)
    except FileNotFoundError:
        logger.error('[%s] Features not found at %s — run Notebook 01 first!', category, feat_path)
        continue
    except Exception as exc:
        logger.error('[%s] Failed to load features: %s', category, exc)
        continue

    # Subsample
    memory_bank = subsample_memory_bank(
        features, CORESET_RATIO, DEVICE,
        checkpoint_path=ckpt_path,
        checkpoint_every=CHECKPOINT_EVERY,
    )

    # Save
    try:
        torch.save(memory_bank, save_path)
        logger.info('[%s] Saved memory bank %s to %s', category, memory_bank.shape, save_path)
    except Exception as exc:
        logger.error('[%s] Failed to save memory bank: %s', category, exc)

logger.info('All categories done!')

## ✅ Summary
At this point, Google Drive should contain:
```
defects-detection-CV/outputs/memory_banks/
   bottle_memory_bank.pt    ← (~1.6k patches, 1536 dims)
   carpet_memory_bank.pt    ← (~2.2k patches, 1536 dims)
   screw_memory_bank.pt     ← (~2.5k patches, 1536 dims)
```
These compressed memory banks are ~100× smaller than the raw features.
The next notebook (`03_memory_bank`) will build FAISS indices on top
of them for ultra-fast nearest-neighbor search at test time.